# ***Data preparation and base evaluation***

In [1]:
import os
os.environ.setdefault("HF_HOME", "/media/caotulab/303A225B3A221DFA/hf_cache")
os.environ.setdefault("WANDB_PROJECT", "alqac-halong-finetune")

import random
import wandb
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

wandb.login()


/media/caotulab/303A225B3A221DFA/envs/nina/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/caotulab/303A225B3A221DFA/Sang_IIPP/tmp/ipykernel_2613695/1385538954.py:8: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/caotulab/.netrc.
wandb: Currently logged in as: syun_1208 (syun12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import json
import os
import random
import re

random.seed(42)

CORPUS_PATH = os.environ.get("ALQAC_CORPUS", "../data/corpus_law_pub.json")
TEST_PATH = os.environ.get("ALQAC_TEST", "../data/ALQAC2026_public_test.json")
EVAL_HELDOUT_CASES = 10
ICT_SENTENCES_PER_ARTICLE = 1
MIN_SENTENCE_LEN = 25

with open(CORPUS_PATH, "r", encoding="utf-8") as handle:
    documents = json.load(handle)

corpus = {}
number_to_aid = {}
for law in documents:
    law_id = str(law.get("law_id", "")).strip()
    article_no = 0
    for article in law.get("content", []):
        article_no += 1
        number_to_aid[(law_id, article_no)] = article["aid"]
        text = str(article.get("content_Article", "")).strip()
        if text:
            corpus[f"{law_id}::{article['aid']}"] = text

CORPUS_LAW_IDS = {cid.split("::")[0] for cid in corpus}

_CODE_RE = re.compile(r"\d+/\d{4}/[A-Za-zĐđ\-]+")
_ART_RE = re.compile(r"[Đđ]iều\s+(\d+)")
_OUTDATED = ("1987", "1993", "1995", "1998", "2000", "2003", "2004", "2005", "2006", "2009")

def _resolve_law_id(name):
    for match in _CODE_RE.finditer(name):
        if match.group(0) in CORPUS_LAW_IDS:
            return match.group(0)
    low = name.lower()
    outdated = any(year in low for year in _OUTDATED)
    if "tố tụng dân sự" in low:
        return "92/2015/QH13"
    if "tố tụng hành chính" in low:
        return "93/2015/QH13"
    if "dân sự" in low:
        return None if outdated else "91/2015/QH13"
    if "hình sự" in low:
        return "100/2015/QH13"
    if "đất đai" in low:
        return None if outdated else "45/2013/QH13"
    if "hôn nhân" in low:
        return None if outdated else "52/2014/QH13"
    if "án phí" in low or "lệ phí" in low:
        return None if "pháp lệnh" in low else "326/2016/UBTVQH14"
    if "thi hành án" in low:
        return "26/2008/QH12"
    if "hộ tịch" in low:
        return "60/2014/QH13"
    if "khiếu nại" in low:
        return None if ("tố cáo" in low or outdated) else "02/2011/QH13"
    if "tổ chức tín dụng" in low:
        return "47/2010/QH12"
    if "kinh doanh bất động sản" in low:
        return None if outdated else "66/2014/QH13"
    if "xây dựng" in low:
        return "50/2014/QH13"
    return None

def _cited_cids(related_text):
    cids = set()
    for line in str(related_text or "").splitlines():
        if "|" not in line:
            continue
        name, remainder = line.split("|", 1)
        law_id = _resolve_law_id(name.strip())
        if law_id is None:
            continue
        for raw_number in _ART_RE.findall(remainder):
            aid = number_to_aid.get((law_id, int(raw_number)))
            cid = f"{law_id}::{aid}" if aid is not None else None
            if cid in corpus:
                cids.add(cid)
    return cids

labelled_cases = []
if os.path.exists(TEST_PATH):
    with open(TEST_PATH, "r", encoding="utf-8") as handle:
        cases = json.load(handle)
    for case in cases:
        query = str(case.get("case_query", "")).strip()
        cids = _cited_cids(case.get("related_law_provisions", ""))
        if query and cids:
            labelled_cases.append((str(case.get("case_id", "")), query, cids))

random.shuffle(labelled_cases)
eval_cases = labelled_cases[:EVAL_HELDOUT_CASES] if EVAL_HELDOUT_CASES > 0 else []
train_cases = labelled_cases[EVAL_HELDOUT_CASES:] if EVAL_HELDOUT_CASES > 0 else labelled_cases

alqac_records = [(q, corpus[cid], cid) for (_id, q, cids) in train_cases for cid in cids]

def _sentences(text):
    parts = re.split(r"(?<=[.;\n])\s+", text)
    return [p.strip() for p in parts if len(p.strip()) >= MIN_SENTENCE_LEN]

ict_records = []
for cid, text in corpus.items():
    sents = _sentences(text)
    if not sents:
        continue
    for sentence in random.sample(sents, min(ICT_SENTENCES_PER_ARTICLE, len(sents))):
        ict_records.append((sentence, text, cid))

train_records = alqac_records + ict_records
random.shuffle(train_records)

if eval_cases:
    queries = {cid_key: query for (cid_key, query, _cids) in eval_cases}
    relevant_docs = {cid_key: set(cids) for (cid_key, _q, cids) in eval_cases}
else:
    random.shuffle(ict_records)
    hold = ict_records[:1000]
    queries = {str(i): a for i, (a, _p, _c) in enumerate(hold)}
    relevant_docs = {str(i): {c} for i, (_a, _p, c) in enumerate(hold)}

print(f"corpus: {len(corpus)} articles | labelled cases: {len(labelled_cases)}")
print(f"train cases: {len(train_cases)} -> supervised {len(alqac_records)} | ICT {len(ict_records)}")
print(f"total train pairs: {len(train_records)} | eval queries: {len(queries)}")


corpus: 3352 articles | labelled cases: 50
train cases: 40 -> supervised 387 | ICT 3352
total train pairs: 3739 | eval queries: 10


In [3]:
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import (
    InformationRetrievalEvaluator,
    SequentialEvaluator,
)
from sentence_transformers.util import cos_sim

model = SentenceTransformer(
    "hiieu/halong_embedding",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
model.max_seq_length = 512
matryoshka_dimensions = [768, 512, 256, 128, 64]
matryoshka_evaluators = []
for dim in matryoshka_dimensions:
    ir_evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=f"dim_{dim}",
        truncate_dim=dim,
        score_functions={"cosine": cos_sim},
    )
    matryoshka_evaluators.append(ir_evaluator)

evaluator = SequentialEvaluator(matryoshka_evaluators)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14021.17it/s]


In [4]:
results = evaluator(model)
for k, v in results.items():
    print(k, v)

dim_768_cosine_accuracy@1 0.1
dim_768_cosine_accuracy@3 0.2
dim_768_cosine_accuracy@5 0.3
dim_768_cosine_accuracy@10 0.4
dim_768_cosine_precision@1 0.1
dim_768_cosine_precision@3 0.1
dim_768_cosine_precision@5 0.1
dim_768_cosine_precision@10 0.08
dim_768_cosine_recall@1 0.004545454545454545
dim_768_cosine_recall@3 0.012545454545454545
dim_768_cosine_recall@5 0.02297326203208556
dim_768_cosine_recall@10 0.05973796791443851
dim_768_cosine_ndcg@10 0.09064292091138462
dim_768_cosine_mrr@10 0.18
dim_768_cosine_map@100 0.03102745221782836
dim_512_cosine_accuracy@1 0.0
dim_512_cosine_accuracy@3 0.1
dim_512_cosine_accuracy@5 0.3
dim_512_cosine_accuracy@10 0.3
dim_512_cosine_precision@1 0.0
dim_512_cosine_precision@3 0.03333333333333333
dim_512_cosine_precision@5 0.08
dim_512_cosine_precision@10 0.06999999999999999
dim_512_cosine_recall@1 0.0
dim_512_cosine_recall@3 0.004
dim_512_cosine_recall@5 0.018427807486631014
dim_512_cosine_recall@10 0.034737967914438506
dim_512_cosine_ndcg@10 0.06098094

# ***Training***

In [5]:
from datasets import Dataset
from sentence_transformers.util import mine_hard_negatives

alqac_dataset = Dataset.from_dict(
    {
        "anchor": [anchor for (anchor, _pos, _cid) in alqac_records],
        "positive": [positive for (_a, positive, _cid) in alqac_records],
    }
)

corpus_documents = list(dict.fromkeys(corpus.values()))

alqac_triplets = mine_hard_negatives(
    alqac_dataset,
    model,
    corpus=corpus_documents,
    num_negatives=5,
    range_min=0,
    range_max=100,
    sampling_strategy="top",
    batch_size=64,
    output_format="triplet",
)

ict_dataset = Dataset.from_dict(
    {
        "anchor": [anchor for (anchor, _pos, _cid) in ict_records],
        "positive": [positive for (_a, positive, _cid) in ict_records],
    }
)

train_dataset = {"alqac": alqac_triplets, "ict": ict_dataset}
print(train_dataset)


Found 40 unique queries out of 387 total queries.
Found an average of 9.675 positives per query.


Computing similarity scores: 100%|██████████| 1/1 [00:00<00:00, 64.34it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count             387          1,935               
Mean           0.2844         0.4277        -0.1433
Median         0.2900         0.4237        -0.1366
Std            0.0801         0.0345         0.0764
Min            0.0229         0.3591        -0.4146
25%            0.2421         0.4077        -0.1886
50%            0.2900         0.4237        -0.1366
75%            0.3377         0.4468        -0.0930
Max            0.4795         0.5832         0.0443
{'alqac': Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1935
}), 'ict': Dataset({
    features: ['anchor', 'positive'],
    num_rows: 3352
})}


In [6]:
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss, MatryoshkaLoss

matryoshka_dimensions = [768, 512, 256, 128, 64]
inner_train_loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16)
train_loss = MatryoshkaLoss(
    model, inner_train_loss, matryoshka_dims=matryoshka_dimensions
)

/media/caotulab/303A225B3A221DFA/Sang_IIPP/tmp/ipykernel_2613695/69537944.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CachedMultipleNegativesRankingLoss, MatryoshkaLoss


In [7]:
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers

OUTPUT_DIR = "/media/caotulab/303A225B3A221DFA/nina_alqac_embedding"

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=100,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    bf16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    multi_dataset_batch_sampler=MultiDatasetBatchSamplers.ROUND_ROBIN,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_dim_768_cosine_ndcg@10",
    report_to="wandb",
    run_name="nina-alqac-embedding",
)


/media/caotulab/303A225B3A221DFA/Sang_IIPP/tmp/ipykernel_2613695/93417490.py:2: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


In [8]:
from sentence_transformers import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

In [9]:
trainer.train()
trainer.save_model()

Step,Training Loss,Validation Loss,Dim 768 Cosine Accuracy@1,Dim 768 Cosine Accuracy@3,Dim 768 Cosine Accuracy@5,Dim 768 Cosine Accuracy@10,Dim 768 Cosine Precision@1,Dim 768 Cosine Precision@3,Dim 768 Cosine Precision@5,Dim 768 Cosine Precision@10,Dim 768 Cosine Recall@1,Dim 768 Cosine Recall@3,Dim 768 Cosine Recall@5,Dim 768 Cosine Recall@10,Dim 768 Cosine Ndcg@10,Dim 768 Cosine Mrr@10,Dim 768 Cosine Map@100,Dim 512 Cosine Accuracy@1,Dim 512 Cosine Accuracy@3,Dim 512 Cosine Accuracy@5,Dim 512 Cosine Accuracy@10,Dim 512 Cosine Precision@1,Dim 512 Cosine Precision@3,Dim 512 Cosine Precision@5,Dim 512 Cosine Precision@10,Dim 512 Cosine Recall@1,Dim 512 Cosine Recall@3,Dim 512 Cosine Recall@5,Dim 512 Cosine Recall@10,Dim 512 Cosine Ndcg@10,Dim 512 Cosine Mrr@10,Dim 512 Cosine Map@100,Dim 256 Cosine Accuracy@1,Dim 256 Cosine Accuracy@3,Dim 256 Cosine Accuracy@5,Dim 256 Cosine Accuracy@10,Dim 256 Cosine Precision@1,Dim 256 Cosine Precision@3,Dim 256 Cosine Precision@5,Dim 256 Cosine Precision@10,Dim 256 Cosine Recall@1,Dim 256 Cosine Recall@3,Dim 256 Cosine Recall@5,Dim 256 Cosine Recall@10,Dim 256 Cosine Ndcg@10,Dim 256 Cosine Mrr@10,Dim 256 Cosine Map@100,Dim 128 Cosine Accuracy@1,Dim 128 Cosine Accuracy@3,Dim 128 Cosine Accuracy@5,Dim 128 Cosine Accuracy@10,Dim 128 Cosine Precision@1,Dim 128 Cosine Precision@3,Dim 128 Cosine Precision@5,Dim 128 Cosine Precision@10,Dim 128 Cosine Recall@1,Dim 128 Cosine Recall@3,Dim 128 Cosine Recall@5,Dim 128 Cosine Recall@10,Dim 128 Cosine Ndcg@10,Dim 128 Cosine Mrr@10,Dim 128 Cosine Map@100,Dim 64 Cosine Accuracy@1,Dim 64 Cosine Accuracy@3,Dim 64 Cosine Accuracy@5,Dim 64 Cosine Accuracy@10,Dim 64 Cosine Precision@1,Dim 64 Cosine Precision@3,Dim 64 Cosine Precision@5,Dim 64 Cosine Precision@10,Dim 64 Cosine Recall@1,Dim 64 Cosine Recall@3,Dim 64 Cosine Recall@5,Dim 64 Cosine Recall@10,Dim 64 Cosine Ndcg@10,Dim 64 Cosine Mrr@10,Dim 64 Cosine Map@100,Sequential Score
50,14.568671,No log,0.000000,0.300000,0.400000,0.400000,0.000000,0.133333,0.140000,0.090000,0.000000,0.018428,0.053310,0.063738,0.095524,0.141667,0.038424,0.000000,0.100000,0.200000,0.300000,0.000000,0.033333,0.080000,0.070000,0.000000,0.004000,0.019765,0.034193,0.063410,0.091667,0.027614,0.100000,0.100000,0.200000,0.300000,0.100000,0.033333,0.060000,0.090000,0.005882,0.005882,0.013882,0.052029,0.082421,0.135000,0.033796,0.000000,0.100000,0.100000,0.500000,0.000000,0.033333,0.040000,0.080000,0.000000,0.005882,0.011765,0.045748,0.065361,0.103175,0.020005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003244,0.000000
100,11.525712,No log,0.100000,0.300000,0.300000,0.300000,0.100000,0.100000,0.080000,0.070000,0.005882,0.022382,0.026382,0.040265,0.083135,0.200000,0.038125,0.100000,0.300000,0.300000,0.300000,0.100000,0.133333,0.080000,0.050000,0.005882,0.026382,0.026382,0.032265,0.069221,0.200000,0.040169,0.400000,0.400000,0.400000,0.500000,0.400000,0.166667,0.120000,0.100000,0.072382,0.084882,0.097382,0.127265,0.187121,0.416667,0.118335,0.100000,0.400000,0.400000,0.500000,0.100000,0.133333,0.080000,0.070000,0.005882,0.072382,0.072382,0.100765,0.111959,0.260000,0.056628,0.000000,0.000000,0.000000,0.100000,0.000000,0.000000,0.000000,0.010000,0.000000,0.000000,0.000000,0.012500,0.008431,0.014286,0.007263,0.008431


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x707eb01c6ad0>> (for post_run_cell), with arguments args (<ExecutionResult object at 707eb01e33d0, execution_count=9 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 707eb01e2990, raw_cell="trainer.train()
trainer.save_model()" transformed_cell="trainer.train()
trainer.save_model()
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bcaotulab_server/home/caotulab/Desktop/Long/ALQAC/notebooks/halong_embedding_fine_tuning.ipynb#X14sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

# ***Re-evaluate after fine-tuning***

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

fine_tuned_model = SentenceTransformer(
    args.output_dir, device="cuda" if torch.cuda.is_available() else "cpu"
)
results = evaluator(fine_tuned_model)
for k, v in results.items():
    print(k, v)

# ***Evaluation test result***

In [ ]:
import numpy as np
import torch

K_VALUES = [1, 3, 5, 10]

corpus_ids = list(corpus.keys())
corpus_texts = [corpus[cid] for cid in corpus_ids]
document_embeddings = fine_tuned_model.encode(
    corpus_texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

query_ids = list(queries.keys())
query_texts = [queries[qid] for qid in query_ids]
query_embeddings = fine_tuned_model.encode(
    query_texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

similarity = query_embeddings @ document_embeddings.T
ranking = torch.topk(similarity, k=max(K_VALUES), dim=1).indices.cpu().numpy()

table = []
for k in K_VALUES:
    precision_at_k, recall_at_k, accuracy_at_k, f1_at_k = [], [], [], []
    for row, qid in enumerate(query_ids):
        relevant = relevant_docs[qid]
        retrieved = [corpus_ids[idx] for idx in ranking[row, :k]]
        hits = sum(1 for cid in retrieved if cid in relevant)
        precision = hits / k
        recall = hits / len(relevant) if relevant else 0.0
        accuracy = 1.0 if hits > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        precision_at_k.append(precision)
        recall_at_k.append(recall)
        accuracy_at_k.append(accuracy)
        f1_at_k.append(f1)
    table.append(
        (
            k,
            float(np.mean(accuracy_at_k)),
            float(np.mean(precision_at_k)),
            float(np.mean(recall_at_k)),
            float(np.mean(f1_at_k)),
        )
    )

print(f"{'K':>3} | {'Accuracy@K':>10} | {'Precision@K':>11} | {'Recall@K':>9} | {'F1@K':>7}")
for k, accuracy, precision, recall, f1 in table:
    print(f"{k:>3} | {accuracy:>10.4f} | {precision:>11.4f} | {recall:>9.4f} | {f1:>7.4f}")


# ***Push to Hugging Face Hub***

In [ ]:
from huggingface_hub import login

HUB_MODEL_ID = "leonpham1208/alqac_halong_embedding"

login()

fine_tuned_model.push_to_hub(
    HUB_MODEL_ID,
    private=True,
    exist_ok=True,
    train_datasets=["ALQAC2026_public_test", "corpus_law_pub"],
)